## CIFAR 10 Frequency Domain CNN

In [ ]:
"""
CIFAR-10 classification on FFT magnitude-spectrum inputs (TensorFlow / Keras)

This script replicates a CIFAR-10 classifier but replaces raw RGB images
with their 2D FFT magnitude spectrum (per-channel). It implements preprocessing,
model definition, training, evaluation, and plotting.

How to use:
1. Install dependencies (in your environment):
   pip install tensorflow matplotlib numpy

2. Run the script in a Jupyter notebook cell or as a .py file.
   (If running as .py, remove plotting.show() blocking or run in an environment that can display plots.)

Notes:
- The example trains for a small number of epochs for quick testing. Increase
  `EPOCHS` for better performance.
- If GPU/TF isn't available, the script will mention that and exit.
- You can switch `PER_CHANNEL` to False to use grayscale magnitude spectra instead of 3-channel.

Reference: frequency-domain CNN ideas from the uploaded fd_cnn_base.pdf (FDC/FDP).
"""

import numpy as np
import time
import os

# Parameters
BATCH_SIZE = 128
EPOCHS = 10  # increase for better results
LEARNING_RATE = 1e-3
PER_CHANNEL = True  # If True: compute magnitude per RGB channel (3-channel input). If False: grayscale (1-channel)
SAVE_PREPROCESSED = True  # Save computed magnitude spectrums to disk to avoid recomputing
DATA_CACHE = "cifar10_mag.npz"


def magnitude_spectrum_rgb(img, per_channel=True):
    """
    Compute magnitude spectrum for an image.
    img: H x W x 3 uint8 or float
    returns H x W x C' (float32) where C' is 3 if per_channel else 1
    """
    H, W, C = img.shape
    if per_channel:
        mag = np.zeros((H, W, 3), dtype=np.float32)
        for ch in range(3):
            channel = img[..., ch].astype(np.float32)
            F = np.fft.fft2(channel)
            Fshift = np.fft.fftshift(F)
            M = np.abs(Fshift)
            M = np.log1p(M)
            M = (M - M.min()) / (M.max() - M.min() + 1e-9)
            mag[..., ch] = M
        return mag
    else:
        # grayscale
        gray = np.mean(img.astype(np.float32), axis=2)
        F = np.fft.fft2(gray)
        Fshift = np.fft.fftshift(F)
        M = np.abs(Fshift)
        M = np.log1p(M)
        M = (M - M.min()) / (M.max() - M.min() + 1e-9)
        return M[..., np.newaxis]


def prepare_dataset_fft(x, per_channel=True, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print("Loading preprocessed dataset from", cache_path)
        with np.load(cache_path) as data:
            return data['x']
    N = x.shape[0]
    example_out = magnitude_spectrum_rgb(x[0], per_channel=per_channel)
    out_shape = (N, example_out.shape[0], example_out.shape[1], example_out.shape[2])
    out = np.zeros(out_shape, dtype=np.float32)
    start = time.time()
    for i in range(N):
        out[i] = magnitude_spectrum_rgb(x[i], per_channel=per_channel)
        if (i+1) % 5000 == 0:
            elapsed = time.time() - start
            print(f"Processed {i+1}/{N} images, time elapsed {elapsed:.1f}s")
    if cache_path:
        np.savez_compressed(cache_path, x=out)
        print("Saved preprocessed data to", cache_path)
    return out


def build_model(input_shape, num_classes=10):
    import tensorflow as tf
    from tensorflow.keras import layers, models # type: ignore

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPool2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.MaxPool2D((2,2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model


def main():
    # Attempt to import TensorFlow
    try:
        import tensorflow as tf
        from tensorflow.keras.utils import to_categorical # type: ignore
    except Exception as e:
        print("TensorFlow is not available in this environment. Please install TensorFlow and re-run.")
        print("Error:", e)
        return

    # Load CIFAR-10
    (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
    y_train = y_train.flatten()
    y_test = y_test.flatten()

    cache_path = DATA_CACHE if SAVE_PREPROCESSED else None
    print("Preparing train magnitude spectra...")
    x_train_mag = prepare_dataset_fft(x_train, per_channel=PER_CHANNEL, cache_path=cache_path)
    print("Preparing test magnitude spectra...")
    if SAVE_PREPROCESSED and os.path.exists(cache_path):
        # load only x from cache
        with np.load(cache_path) as data:
            x_test_mag = prepare_dataset_fft(x_test, per_channel=PER_CHANNEL, cache_path=None)
    else:
        x_test_mag = prepare_dataset_fft(x_test, per_channel=PER_CHANNEL, cache_path=None)

    input_shape = x_train_mag.shape[1:]
    num_classes = 10

    # Convert labels
    y_train_cat = to_categorical(y_train, num_classes)
    y_test_cat = to_categorical(y_test, num_classes)

    model = build_model(input_shape=input_shape, num_classes=num_classes)
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.summary()

    # Train
    history = model.fit(
        x_train_mag, y_train_cat,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2
    )

    # Evaluate
    test_loss, test_acc = model.evaluate(x_test_mag, y_test_cat, verbose=0)
    print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")

    # Plot
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='train_loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['accuracy'], label='train_acc')
    plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()

    plt.tight_layout()
    plt.show()


if __name__ == '__main__':
    main()